In [26]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time

# --- Image config ---
IMAGE_PATH = "target_image.jpg"
OUTPUT_PATH = "edited_output.jpg"


In [27]:
# left options
tools = [
    "Exposure", "Contrast", "Sharpness", "Saturation", 
    "Temperature", "Vignette"
]

In [28]:
state = {tool: 50 for tool in tools}
active_tool = "Exposure"

# Variables for Save cooldown
last_save_time = 0

# --- MediaPipe Setup ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils

In [29]:
# --- Image Processing Functions ---
def apply_filters(img, state):
    result = img.copy()
    
    # 1. Exposure & Contrast
    exposure_beta = (state["Exposure"] - 50) * 2  
    contrast_alpha = state["Contrast"] / 50.0     
    result = cv2.convertScaleAbs(result, alpha=contrast_alpha, beta=exposure_beta)
    
    # 2. Saturation
    if state["Saturation"] != 50:
        hsv = cv2.cvtColor(result, cv2.COLOR_BGR2HSV).astype(np.float32)
        sat_scale = state["Saturation"] / 50.0
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * sat_scale, 0, 255)
        result = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        
    # 3. Temperature 
    if state["Temperature"] != 50:
        temp_shift = (state["Temperature"] - 50)
        B, G, R = cv2.split(result)
        R = cv2.add(R, temp_shift)
        B = cv2.subtract(B, temp_shift)
        result = cv2.merge((B, G, R))
        
    # 4. Sharpness
    if state["Sharpness"] != 50:
        blur = cv2.GaussianBlur(result, (0, 0), 3)
        weight = state["Sharpness"] / 30.0
        result = cv2.addWeighted(result, weight, blur, 1 - weight, 0)
        
    # 5. Vignette (Black > 50, White < 50)
    if state["Vignette"] != 50:
        rows, cols = result.shape[:2]
        
        # Calculate intensity (0.0 to 1.0) whether moving left or right
        if state["Vignette"] > 50:
            intensity = (state["Vignette"] - 50)  / 50.0
        else:
            intensity = (50 - state["Vignette"]) / 50.0
            
        # Determine the spread of the vignette
        sigma_x = cols / (2 + intensity * 5)
        sigma_y = rows / (2 + intensity * 5)
        
        kernel_x = cv2.getGaussianKernel(cols, sigma_x)
        kernel_y = cv2.getGaussianKernel(rows, sigma_y)
        
        # Create and normalize the mask
        kernel = kernel_y * kernel_x.T
        mask = kernel / kernel.max()
        mask = cv2.merge([mask, mask, mask])
        
        if state["Vignette"] > 50:
            # Black Vignette: multiply by mask (edges fade to 0 / black)
            result = (result * mask).astype(np.uint8)
        else:
            # White Vignette: blend edges toward 255 (white)
            result = (result * mask + 255 * (1 - mask)).astype(np.uint8)

    return result


In [30]:
def main():
    global active_tool, last_save_time
    
    # Load Image
    if not os.path.exists(IMAGE_PATH):
        print(f"Error: Could not find '{IMAGE_PATH}'. Please place an image in the directory.")
        return
        
    original_image = cv2.imread(IMAGE_PATH)
    if original_image.shape[1] > 800:
        scale = 800 / original_image.shape[1]
        original_image = cv2.resize(original_image, (0,0), fx=scale, fy=scale)
        
    display_image = original_image.copy()
    
    # Start Webcam
    cap = cv2.VideoCapture(0)
    cap.set(3, 1920) # Width
    cap.set(4, 1080)


    print("App started. Press 'q' to quit.")

    while True:
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        h, w, c = frame.shape

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_frame)
        
        ui_canvas = np.zeros_like(frame)

        # ---------------- UI DRAWING ----------------

        # 1. Draw Left Menu (Tools only)
        menu_y_start = 50
        box_height = 60
        for i, tool in enumerate(tools):
            y1 = menu_y_start + (i * box_height)
            y2 = y1 + box_height - 10
            color = (0, 255, 0) if tool == active_tool else (200, 200, 200)
            cv2.rectangle(ui_canvas, (20, y1), (200, y2), color, cv2.FILLED)
            cv2.putText(ui_canvas, tool, (30, y1 + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

        # 2. Draw Top-Right Save Button
        save_x1, save_y1 = w - 160, 20
        save_x2, save_y2 = w - 20, 80
        cv2.rectangle(ui_canvas, (save_x1, save_y1), (save_x2, save_y2), (0, 150, 255), cv2.FILLED)
        cv2.putText(ui_canvas, "SAVE", (save_x1 + 30, save_y1 + 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # 3. Draw Bottom Slider 
        slider_x1, slider_x2 = 300, 1000
        slider_y = h - 100
        cv2.line(ui_canvas, (slider_x1, slider_y), (slider_x2, slider_y), (255, 255, 255), 5)
        
        knob_x = int(np.interp(state[active_tool], [0, 100], [slider_x1, slider_x2]))
        cv2.circle(ui_canvas, (knob_x, slider_y), 15, (0, 128, 255), cv2.FILLED)
        cv2.putText(ui_canvas, f"Value: {int(state[active_tool])}", (slider_x1, slider_y + 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # ---------------- HAND TRACKING LOGIC ----------------
        if results.multi_hand_landmarks:
            for hand_idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
                hand_label = results.multi_handedness[hand_idx].classification[0].label
                
                idx_x = int(hand_landmarks.landmark[8].x * w)
                idx_y = int(hand_landmarks.landmark[8].y * h)
                
                mp_draw.draw_landmarks(ui_canvas, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
                # --- LEFT HAND: Menu Selection ---
                if hand_label == "Left":
                    cv2.circle(ui_canvas, (idx_x, idx_y), 10, (255, 0, 0), cv2.FILLED)
                    if 20 < idx_x < 200:
                        for i, tool in enumerate(tools):
                            y1 = menu_y_start + (i * box_height)
                            y2 = y1 + box_height - 10
                            if y1 < idx_y < y2:
                                active_tool = tool

                # --- RIGHT HAND: Save Button & Slider Control ---
                elif hand_label == "Right":
                    cv2.circle(ui_canvas, (idx_x, idx_y), 10, (0, 0, 255), cv2.FILLED)
                    
                    # 1. Check for Save Button Interaction (Top Right)
                    if save_x1 < idx_x < save_x2 and save_y1 < idx_y < save_y2:
                        # Only save if 2 seconds have passed since the last save
                        if time.time() - last_save_time > 2.0:
                            cv2.imwrite(OUTPUT_PATH, display_image)
                            print(f"Saved successfully to {OUTPUT_PATH}!")
                            last_save_time = time.time()
                            
                    # 2. Check for Slider Interaction (Bottom)
                    elif h - 200 < idx_y < h:
                        new_val = np.interp(idx_x, [slider_x1, slider_x2], [0, 100])
                        state[active_tool] = new_val

        # Apply filters based on current state
        # Apply filters based on current state
        display_image = apply_filters(original_image, state)

        # Draw "SAVED!" notification text ONLY on the edited image
        if time.time() - last_save_time < 2:
            img_h, img_w = display_image.shape[:2]
            # Center the text dynamically based on the image size
            cv2.putText(display_image, "SAVED!", (img_w//2 - 150, img_h//2), 
                        cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 5)

        # Display windows
        cv2.namedWindow("Hand Control Interface", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("Hand Control Interface",1920,1080 )
        cv2.imshow("Hand Control Interface", ui_canvas)
        
        cv2.namedWindow("Image Editor View", cv2.WINDOW_NORMAL)

        # 2. Set the exact resolution of the window (Width, Height)
        cv2.resizeWindow("Image Editor View", 800, 550)
        cv2.imshow("Image Editor View", display_image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

App started. Press 'q' to quit.
Saved successfully to edited_output.jpg!
Saved successfully to edited_output.jpg!
